# 10 · Window Functions

Window functions compute across a set of rows **related to the current row**
without collapsing them (unlike `GROUP BY`). The magic word is `OVER`.
- `ROW_NUMBER`, `RANK`, `DENSE_RANK`
- `PARTITION BY` (windows per group) and `ORDER BY` inside `OVER`
- running totals with `SUM() OVER (...)`
- `LAG` / `LEAD` (previous / next row)

> Requires SQLite 3.25+ (the version bundled with modern Python — you're fine).

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## `ROW_NUMBER` — number rows
Rank products by price, most expensive = 1:

In [ ]:
%%sql
SELECT product_name, unit_price,
       ROW_NUMBER() OVER (ORDER BY unit_price DESC) AS price_rank
FROM products
LIMIT 10;

## `PARTITION BY` — restart per group
Rank products **within each category** by price. The partition restarts the
numbering for every category.

In [ ]:
%%sql
SELECT category_id, product_name, unit_price,
       RANK() OVER (PARTITION BY category_id ORDER BY unit_price DESC) AS rank_in_cat
FROM products
ORDER BY category_id, rank_in_cat;

## RANK vs DENSE_RANK vs ROW_NUMBER
These three only differ **when there are ties** — rows with the same `ORDER BY`
value. Our product prices are all distinct, so to see the difference clearly we
use a small scoreboard with deliberate ties (Ana & Ben both 95, Cyd & Dan both
88):

In [ ]:
%%sql
WITH scores(player, score) AS (
    VALUES ('Ana', 95), ('Ben', 95), ('Cyd', 88), ('Dan', 88), ('Eve', 70)
)
SELECT player, score,
       ROW_NUMBER() OVER (ORDER BY score DESC) AS row_number,
       RANK()       OVER (ORDER BY score DESC) AS rank,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rank
FROM scores;

Read the tied rows to see exactly how they differ:

| player | score | row_number | rank | dense_rank |
|--------|------:|:----------:|:----:|:----------:|
| Ana | 95 | 1 | 1 | 1 |
| Ben | 95 | 2 | 1 | 1 |
| Cyd | 88 | 3 | 3 | 2 |
| Dan | 88 | 4 | 3 | 2 |
| Eve | 70 | 5 | 5 | 3 |

- **`ROW_NUMBER`** — always unique, ties broken arbitrarily: `1, 2, 3, 4, 5`.
- **`RANK`** — ties share a number, then it **skips** (leaves a gap): `1, 1, 3, 3, 5`.
- **`DENSE_RANK`** — ties share a number, **no gaps**: `1, 1, 2, 2, 3`.

Rule of thumb: use `ROW_NUMBER` to pick exactly one row per group, `RANK` for
"standard competition" ranking (two golds → no silver), and `DENSE_RANK` when you
don't want gaps in the numbering.

## Running total
Order revenue over time, plus a cumulative total. The frame defaults to all rows
from the start up to the current row when you add `ORDER BY`.

In [ ]:
%%sql
WITH order_totals AS (
    SELECT o.order_id, o.order_date,
           SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.order_date
)
SELECT order_date, order_id, order_total,
       SUM(order_total) OVER (ORDER BY order_date, order_id) AS running_total
FROM order_totals
ORDER BY order_date, order_id;

## `LAG` / `LEAD`
Compare each order's total to the previous order's total.

In [ ]:
%%sql
WITH order_totals AS (
    SELECT o.order_id, o.order_date,
           SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.order_date
)
SELECT order_date, order_total,
       LAG(order_total)  OVER (ORDER BY order_date, order_id) AS prev_total,
       order_total - LAG(order_total) OVER (ORDER BY order_date, order_id) AS change
FROM order_totals
ORDER BY order_date, order_id;

## Average per group alongside each row
`AVG() OVER (PARTITION BY ...)` shows the category average next to each product without collapsing rows:

In [ ]:
%%sql
SELECT category_id, product_name, unit_price,
       ROUND(AVG(unit_price) OVER (PARTITION BY category_id), 2) AS cat_avg
FROM products
ORDER BY category_id, unit_price DESC
LIMIT 12;

## Practice

**✏️ Exercise 1.** Number the customers by signup order (1 = earliest) using ROW_NUMBER.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT first_name, signup_date,
       ROW_NUMBER() OVER (ORDER BY signup_date) AS signup_order
FROM customers;

**✏️ Exercise 2.** Within each category, find the single most expensive product (use ROW_NUMBER in a CTE and keep rank = 1).

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH ranked AS (
  SELECT category_id, product_name, unit_price,
         ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY unit_price DESC) AS rn
  FROM products
)
SELECT category_id, product_name, unit_price
FROM ranked WHERE rn = 1
ORDER BY category_id;

### ✅ Recap
Window functions rank, number, and accumulate across related rows while keeping
every row. `PARTITION BY` groups the window; `ORDER BY` inside `OVER` orders it;
`LAG`/`LEAD` look at neighbors.

**Next:** `11_ctes.ipynb`.